# LA 02 — Causal second-touch candidates and flow features

**Status:** exploratory feature extraction, not a trading strategy backtest. The data-readiness audit (LA 01 v0.1.2) has passed for BTC/ETH historical Binance USD-M perpetual candles and sampled aggregated trades. This notebook constructs **price-defined** candidate events, then attaches delta and traded-volume-at-price observations available by the *end of the second rejection candle*. It does **not** calculate post-entry returns, stops, targets, profitability, or claim causality of hidden orders.

Price-zone anchors are proxies: the **previous completed UTC day high/low** and the latest **fully confirmed five-minute pivot**. They are **not** yet volume-profile-derived discretionary levels. Research parameters are declared in code and saved to `02_feature_contract.json`; they have **not** been optimized or formally frozen for trading evaluation. Trade-volume-at-level uses **1bp price bins**, so it is approximate.

In [1]:
from pathlib import Path
import sys
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / "src" / "data_engineering").exists():
    if REPO == REPO.parent:
        raise RuntimeError("Launch Jupyter from repository or a subdirectory")
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.data_engineering.modeling.liquidity_absorption_stage_a import binance_root
from src.data_engineering.modeling.liquidity_absorption_stage_a02 import Config, LA_A02_VERSION, run_a02
assert LA_A02_VERSION == "LA-A02-0.1.0", "Pull the latest repository and restart the kernel"
DATA_ROOT = binance_root(REPO)
OUT = REPO / "notebooks" / "Liquidity_Absorption" / "02"
CFG = Config()
print("A02:", LA_A02_VERSION, " | Binance:", DATA_ROOT, " | Outputs:", OUT)
print("Candidate dates:", CFG.start, "to", CFG.end)
print("Reminder: EXPLORATORY FEATURE GENERATION; NO P&L")

A02: LA-A02-0.1.0  | Binance: D:\MarketData\Binance  | Outputs: D:\Projects\deribit_market_data\notebooks\Liquidity_Absorption\02
Candidate dates: 2026-07-01 to 2026-09-30
Reminder: EXPLORATORY FEATURE GENERATION; NO P&L


In [2]:
result = run_a02(DATA_ROOT, OUT, cfg=CFG)
result

{'version': 'LA-A02-0.1.0',
 'status': 'FEATURE_EXTRACTION_ONLY',
 'symbols': ['BTCUSDT', 'ETHUSDT'],
 'summary': [{'symbol': 'BTCUSDT',
   'minute_rows_including_warmup': 136800,
   'price_only_candidates': 212,
   'profile_present': 212,
   'missing_trade_day_events': 0,
   'first_signal': '2026-07-01 02:33:00+00:00',
   'last_signal': '2026-09-30 20:58:00+00:00'},
  {'symbol': 'ETHUSDT',
   'minute_rows_including_warmup': 136800,
   'price_only_candidates': 240,
   'profile_present': 240,
   'missing_trade_day_events': 0,
   'first_signal': '2026-07-01 02:52:00+00:00',
   'last_signal': '2026-09-30 20:37:00+00:00'}],
 'caution': 'No backtested P&L, economic edge, stop simulation, prospective holdout or proof of absorption.'}

In [3]:
display(pd.read_csv(OUT / "02_coverage.csv"))
levels = pd.read_csv(OUT / "02_price_only_candidates.csv")
features = pd.read_csv(OUT / "02_causal_flow_features.csv")
print("Price-defined candidates:", len(levels), " | Enriched rows:", len(features))
assert len(levels) == len(features), "Candidate/enrichment mismatch"
assert not ({"delta_5m", "delta_15m", "cvd_utc_session", "pnl", "future_return"} & set(levels.columns)), "Price-only file contains nonprice features"
if len(levels):
    display(levels.groupby(["symbol", "side", "level_type"]).size().rename("candidates").reset_index())
    display(levels.head(10))
if len(features):
    display(features.groupby(["symbol", "profile_status"]).size().rename("events").reset_index())
    display(features[["symbol", "side", "signal_time_utc", "level_price", "delta_5m", "delta_15m", "zone_quote_last5m", "zone_quote_prev5m", "near_zone_share_last5m", "profile_status"]].head(10))

,symbol,minute_rows_including_warmup,price_only_candidates,profile_present,missing_trade_day_events,first_signal,last_signal
0,BTCUSDT,136800,212,212,0,2026-07-01 02:33:00+00:00,2026-09-30 20:58:00+00:00
1,ETHUSDT,136800,240,240,0,2026-07-01 02:52:00+00:00,2026-09-30 20:37:00+00:00


Price-defined candidates: 452  | Enriched rows: 452


,symbol,side,level_type,candidates
0,BTCUSDT,long,pivot_5m,85
1,BTCUSDT,long,prior_day,13
2,BTCUSDT,short,pivot_5m,91
3,BTCUSDT,short,prior_day,23
4,ETHUSDT,long,pivot_5m,113
5,ETHUSDT,long,prior_day,17
6,ETHUSDT,short,pivot_5m,84
7,ETHUSDT,short,prior_day,26


,symbol,side,level_type,level_source_time,level_price,zone_halfwidth,first_touch_close,second_touch_close,signal_time_utc,bar_close,bar_low,bar_high,bar_open,lower_wick_fraction,upper_wick_fraction,first_touch_low,first_touch_high,minutes_between_touches
0,BTCUSDT,short,pivot_5m,2026-07-01 01:50:00+00:00,58991.6,11.79832,2026-07-01 02:29:00+00:00,2026-07-01 02:33:00+00:00,2026-07-01 02:33:00+00:00,58935.8,58887.9,58984.5,58922.6,0.359213,0.504141,58928.6,58982.7,4.0
1,BTCUSDT,long,pivot_5m,2026-07-01 12:15:00+00:00,58350.9,11.67018,2026-07-01 12:41:00+00:00,2026-07-01 12:44:00+00:00,2026-07-01 12:44:00+00:00,58375.2,58344.2,58380.0,58358.3,0.393855,0.134078,58351.2,58386.9,3.0
2,BTCUSDT,long,pivot_5m,2026-07-01 15:40:00+00:00,60099.1,12.01982,2026-07-01 15:59:00+00:00,2026-07-01 16:11:00+00:00,2026-07-01 16:11:00+00:00,60121.1,60095.1,60139.3,60122.8,0.588235,0.373303,60106.0,60115.0,12.0
3,BTCUSDT,short,prior_day,2026-07-01 00:00:00+00:00,61322.0,12.26440,2026-07-02 10:43:00+00:00,2026-07-02 10:55:00+00:00,2026-07-02 10:55:00+00:00,61285.9,61266.0,61311.8,61266.0,0.000000,0.565502,61269.9,61330.0,12.0
4,BTCUSDT,short,pivot_5m,2026-07-02 10:45:00+00:00,61330.0,12.26600,2026-07-02 11:10:00+00:00,2026-07-02 11:24:00+00:00,2026-07-02 11:24:00+00:00,61296.9,61269.9,61330.0,61270.0,0.001664,0.550749,61263.9,61324.9,14.0
5,BTCUSDT,short,prior_day,2026-07-01 00:00:00+00:00,61322.0,12.26440,2026-07-02 11:25:00+00:00,2026-07-02 11:39:00+00:00,2026-07-02 11:39:00+00:00,61268.7,61260.0,61320.9,61262.0,0.032841,0.857143,61280.6,61313.2,14.0
6,BTCUSDT,short,prior_day,2026-07-01 00:00:00+00:00,61322.0,12.26440,2026-07-02 12:11:00+00:00,2026-07-02 12:18:00+00:00,2026-07-02 12:18:00+00:00,61278.1,61275.3,61329.8,61299.6,0.051376,0.554128,61288.8,61323.6,7.0
7,BTCUSDT,long,pivot_5m,2026-07-03 03:55:00+00:00,61327.2,12.26544,2026-07-03 04:10:00+00:00,2026-07-03 04:19:00+00:00,2026-07-03 04:19:00+00:00,61346.3,61334.0,61355.7,61345.6,0.534562,0.433180,61325.3,61360.0,9.0
8,BTCUSDT,long,pivot_5m,2026-07-04 10:40:00+00:00,62436.2,12.48724,2026-07-04 10:52:00+00:00,2026-07-04 11:03:00+00:00,2026-07-04 11:03:00+00:00,62463.1,62440.6,62463.4,62450.1,0.416667,0.013158,62448.3,62470.5,11.0
9,BTCUSDT,short,pivot_5m,2026-07-04 13:00:00+00:00,62671.5,12.53430,2026-07-04 13:23:00+00:00,2026-07-04 13:26:00+00:00,2026-07-04 13:26:00+00:00,62650.0,62642.8,62685.5,62642.9,0.002342,0.831382,62629.3,62688.0,3.0


,symbol,profile_status,events
0,BTCUSDT,PRESENT,212
1,ETHUSDT,PRESENT,240


,symbol,side,signal_time_utc,level_price,delta_5m,delta_15m,zone_quote_last5m,zone_quote_prev5m,near_zone_share_last5m,profile_status
0,BTCUSDT,short,2026-07-01 02:33:00+00:00,58991.6,3.677731e+07,8.908338e+07,4.053905e+07,4.303873e+07,0.386986,PRESENT
1,BTCUSDT,long,2026-07-01 12:44:00+00:00,58350.9,9.246138e+05,4.151690e+06,1.500268e+07,0.000000e+00,0.403512,PRESENT
2,BTCUSDT,long,2026-07-01 16:11:00+00:00,60099.1,-4.689152e+06,-4.515111e+06,4.319031e+06,1.422651e+07,0.121104,PRESENT
3,BTCUSDT,short,2026-07-02 10:55:00+00:00,61322.0,-1.572679e+07,-3.620944e+07,9.087185e+05,0.000000e+00,0.017684,PRESENT
4,BTCUSDT,short,2026-07-02 11:24:00+00:00,61330.0,-7.066809e+06,-4.481281e+07,1.754750e+06,0.000000e+00,0.020499,PRESENT
5,BTCUSDT,short,2026-07-02 11:39:00+00:00,61322.0,4.141116e+06,-2.319864e+07,5.135354e+06,3.870305e+06,0.095278,PRESENT
6,BTCUSDT,short,2026-07-02 12:18:00+00:00,61322.0,-1.130002e+07,-6.129710e+07,2.335722e+07,2.386110e+06,0.297535,PRESENT
7,BTCUSDT,long,2026-07-03 04:19:00+00:00,61327.2,1.502582e+06,-5.928134e+06,2.521030e+06,8.077271e+06,0.208733,PRESENT
8,BTCUSDT,long,2026-07-04 11:03:00+00:00,62436.2,-1.742423e+06,2.413331e+06,1.473748e+06,0.000000e+00,0.152129,PRESENT
9,BTCUSDT,short,2026-07-04 13:26:00+00:00,62671.5,-2.631424e+06,1.423312e+07,4.959962e+06,3.601463e+06,0.241900,PRESENT


## Review before Stage A03

- Verify event timestamps use the **one-minute candle close** and reference levels were known at or before that time.
- Check how many events each source produces, including duplicates/overlapping levels; do not silently combine them.
- Inspect `profile_status`. `MISSING_TRADE_DAY` / `MISSING_TRADE_MINUTE` mean **no usable profile observation**, not zero volume. `PRESENT` means the preceding 10 minutes were available; no economic edge is inferred.
- Level proximity uses a **2bp placeholder** for crypto, not your MNQ 2–4 tick observation. The price-only levels are pragmatic proxies, not a faithful reconstruction of discretionary volume-profile zones.
- No fixed stop, +10 MNQ point break-even, structural trailing stops, profit targets or executable fills are modeled here. Those belong in Stage A03 once this feature contract is reviewed.

Commit/push this executed notebook and the CSV/JSON in `notebooks/Liquidity_Absorption/02/`. A full 92-day trade scan is substantial but uses Parquet batches to limit working memory. **Do not use any future-return or P&L information to retroactively tune the event definitions.**